Overview: 
1. Gets data from yahoo finance of 20 semiconductor tickers, pulling 2 years worth of data
2. Merge into one csv file
3. Add an average volume of the past 20 days (rolling) column and also adds a week number to help with book keeping
4. Has a sanity check at the end to see anny missing data and to check if everything is working smoothly

In [ ]:
#Imports + adding tickers
import yfinance as yf
import pandas as pd
import numpy as np

TICKERS = [
    "NVDA", "AMD", "INTC", "TSM", "AVGO", "QCOM", "TXN", "MU", "ASML", "LRCX",
    "AMAT", "KLAC", "ADI", "MRVL", "ON", "SWKS", "MCHP", "NXPI", "STM", "MPWR",
]

LOOKBACK_PERIOD = "2y"       
ROLLING_WINDOW = 20         
OUTPUT_CSV = "semiconductor_data.csv"

#Flagging a ticker if it's missing too much data (this is the threshold)
MAX_MISSING_FRACTION = 0.05

print(f"Tracking {len(TICKERS)} tickers over the last {LOOKBACK_PERIOD}.")

Tracking 20 tickers over the last 2y.


In [ ]:
#Download all the tickers and adjusts the stock for potential stock splits or dividends paid out
raw = yf.download(
    tickers=TICKERS,
    period=LOOKBACK_PERIOD,
    auto_adjust=True,
    group_by="ticker",
    threads=True,
    progress=False,
)

print(f"Downloaded raw data shape: {raw.shape}")
print(f"Date range: {raw.index.min().date()} to {raw.index.max().date()}")

Downloaded raw data shape: (502, 100)
Date range: 2024-09-03 to 2026-09-02


In [ ]:
#Taking the raw data and turning it into a clean table
frames = []
failed_tickers = []

for ticker in TICKERS:
    if ticker not in raw.columns.get_level_values(0):
        failed_tickers.append(ticker)
        continue
    
    #Extracts and cleaning each ticker
    df = raw[ticker].copy()
    df.columns = [c.lower() for c in df.columns]  

    #Failure check if the ticker is empty
    if df["close"].isna().all():
        failed_tickers.append(ticker)
        continue
    
    df["ticker"] = ticker
    df = df.reset_index().rename(columns={"Date": "date"})
    frames.append(df)
    
#For missing data
if failed_tickers:
    print(f"WARNING: no usable data for: {failed_tickers}")
else:
    print("All 20 tickers returned data.")
    
#Combining the data into one big table
long_df = pd.concat(frames, ignore_index=True)
long_df = long_df[["date", "ticker", "open", "high", "low", "close", "volume"]]
print(f"Long-format shape before cleaning: {long_df.shape}")

All 20 tickers returned data.
Long-format shape before cleaning: (10040, 7)


In [ ]:
#Build calendar to collect the trading days
full_calendar = pd.Index(sorted(long_df["date"].unique()), name="date")


aligned_frames = []
gap_report = {}

#Looping through the tickers except for the failed ones
for ticker in TICKERS:
    if ticker in failed_tickers:
        continue
    #Reindexing to fit the calendar
    tdf = long_df[long_df["ticker"] == ticker].set_index("date")
    tdf = tdf.reindex(full_calendar)
    tdf["ticker"] = ticker

    #Report any missing data
    missing_frac = tdf["close"].isna().mean()
    gap_report[ticker] = missing_frac
    if missing_frac > MAX_MISSING_FRACTION:
        print(f"WARNING: {ticker} is missing {missing_frac:.1%} of trading days "
              f"(threshold {MAX_MISSING_FRACTION:.0%}) -- check this ticker.")

    aligned_frames.append(tdf.reset_index())

#Combine all tickers into one table
aligned_df = pd.concat(aligned_frames, ignore_index=True)
aligned_df = aligned_df.sort_values(["ticker", "date"]).reset_index(drop=True)

fill_cols = ["open", "high", "low", "close", "volume"]
aligned_df[fill_cols] = aligned_df.groupby("ticker")[fill_cols].ffill(limit=3)

print(f"Aligned shape: {aligned_df.shape}")

Aligned shape: (10040, 7)


In [ ]:
#Computing the rolling 20 day volume average
aligned_df = aligned_df.sort_values(["ticker", "date"])
aligned_df["avg_volume_20d"] = (
    aligned_df.groupby("ticker")["volume"]
    .transform(lambda v: v.rolling(window=ROLLING_WINDOW, min_periods=1).mean())
)

#Computing the week number
week_start = aligned_df["date"] - pd.to_timedelta(aligned_df["date"].dt.weekday, unit="D")
unique_weeks = sorted(week_start.unique())
week_lookup = {monday: i + 1 for i, monday in enumerate(unique_weeks)}
aligned_df["week_number"] = week_start.map(week_lookup)

print(f"Weeks covered: {aligned_df['week_number'].min()} to {aligned_df['week_number'].max()}")

Weeks covered: 1 to 105


In [ ]:
#Finalizing the columns and saving
final_df = aligned_df[[
    "date", "ticker", "open", "high", "low", "close", "volume",
    "avg_volume_20d", "week_number",
]].copy()

final_df["date"] = pd.to_datetime(final_df["date"]).dt.strftime("%Y-%m-%d")
final_df = final_df.sort_values(["date", "ticker"]).reset_index(drop=True)

final_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(final_df)} rows to {OUTPUT_CSV}")

Saved 10040 rows to semiconductor_data.csv


In [9]:
#Doing a check to see everyting ran correctly
print("Sample output")

#If any tickers failed
if failed_tickers:
    print(f"Tickers with no data at all: {failed_tickers}")
else:
    print(f"All {len(TICKERS)} tickers present.")

#Checking if any tickers have more missing data than the threshold
bad_gap_tickers = {t: f"{f:.1%}" for t, f in gap_report.items() if f > MAX_MISSING_FRACTION}
if bad_gap_tickers:
    print(f"Tickers with >{MAX_MISSING_FRACTION:.0%} missing trading days: {bad_gap_tickers}")
else:
    print(f"No ticker missing more than {MAX_MISSING_FRACTION:.0%} of trading days.")

#Counts of NaNs
nan_counts = final_df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) > 0:
    print(f"Columns still containing NaNs:\n{nan_counts}")
else:
    print("No NaNs in final output.")

#Counts number of duplicates
dupes = final_df.duplicated(subset=["date", "ticker"]).sum()
print(f"[{'OK' if dupes == 0 else 'FAIL'}] Duplicate (date, ticker) rows: {dupes}")

Sample output
All 20 tickers present.
No ticker missing more than 5% of trading days.
No NaNs in final output.
[OK] Duplicate (date, ticker) rows: 0
